# Regressione delta_skill_albedo vs delta_cover

Terzo pezzo del quadro di convergenza discusso: `03-cover_tas_regression_hybrid`
mostra una relazione tra `delta_cover` e `delta_skill_tas` in Russia-Cina-Siberia
(coerente tra cvh e cvl, coerente tra lead-year, piu' forte su medie
pluriennali - tipico delle simulazioni decadali). La stessa regione mostra una
relazione forte anche tra `delta_skill_albedo` e `delta_skill_tas`. Questo
notebook completa il triangolo: `delta_cover` vs `delta_skill_albedo`.

**A differenza della cover, l'albedo ha un'osservazione indipendente vera**
(GLASS): niente disegno ibrido necessario qui, `delta_skill_albedo` e' un
miglioramento di skill genuino sia per SENS sia per CTRL (stesso schema pulito
di 06-covariance_albedo.ipynb/07-covariance_regression.ipynb, applicato
direttamente).

```
X = delta_cover         = cover_SENS - cover_CTRL (anomalia, non circolare)
Y = delta_skill_albedo  = skill_albedo_SENS - skill_albedo_CTRL (vs GLASS, genuino)
```

Stesse tre mappe (Pearson slope, Pearson r, Spearman rho) + scatter sul box
Siberia di `03-cover_tas_regression_hybrid`, stessa maschera a soglia fissa
`1e-3` su `delta_cover` (vedi quel notebook per la motivazione).


In [ ]:
# rende config.py (in notebooks/) importabile anche da questa sottocartella
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025

exp_ctrl = 'a1ua'
exp_sens = 'a52o'
variables = ['cvh', 'cvl']
SAVE_PATH = str(FIG_DIR / "04_cover_albedo")  # sottocartella dedicata a questo notebook
os.makedirs(SAVE_PATH, exist_ok=True)


In [ ]:
# La logica di calcolo sta in cover_tas_lib.py (stesso modulo di 01/02/03,
# nuova funzione run_one_cover_albedo). Processi spawn freschi, stesso pattern
# robusto adottato in questa sessione.
sys.path.insert(0, os.getcwd())
from cover_tas_lib import run_one_cover_albedo, LEADS


In [ ]:
%%time
import multiprocessing as mp

jobs = [(exp_ctrl, exp_sens, var, y1, y2, SAVE_PATH) for var in variables for (y1, y2) in LEADS]

with mp.get_context('spawn').Pool(processes=2, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(run_one_cover_albedo, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)
